**Name:** Dhruv Marwal

**Class:** BTech Ai, Batch 2

**Roll no:** I043

In [31]:
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

In [32]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/priyanshu2304/hindi-english-truncated-corpus/Hindi_English_Truncated_Corpus.csv")
print(df.columns)
df.head()

Index(['source', 'english_sentence', 'hindi_sentence'], dtype='object')


,source,english_sentence,hindi_sentence
0,ted,politicians do not have permission to do what ...,"राजनीतिज्ञों के पास जो कार्य करना चाहिए, वह कर..."
1,ted,"I'd like to tell you about one such child,",मई आपको ऐसे ही एक बच्चे के बारे में बताना चाहू...
2,indic2012,This percentage is even greater than the perce...,यह प्रतिशत भारत में हिन्दुओं प्रतिशत से अधिक है।
3,ted,what we really mean is that they're bad at not...,हम ये नहीं कहना चाहते कि वो ध्यान नहीं दे पाते
4,indic2012,.The ending portion of these Vedas is called U...,इन्हीं वेदों का अंतिम भाग उपनिषद कहलाता है।


In [36]:
def filter_len(df, min_len=2, max_len=20):
    df['eng_len'] = df['english_sentence'].apply(lambda x: len(str(x).split()))
    df['hin_len'] = df['hindi_sentence'].apply(lambda x: len(str(x).split()))
    
    df = df[
        (df['eng_len'] >= min_len) & (df['eng_len'] <= max_len) &
        (df['hin_len'] >= min_len) & (df['hin_len'] <= max_len)
    ]
    
    df = df.drop(columns=['eng_len', 'hin_len'])
    return df

df = df[['english_sentence', 'hindi_sentence']]   # keep required columns
df = df.dropna()                                 # remove nulls

df = df[
    (df['english_sentence'].str.len() < 150) &    # remove very long chars
    (df['hindi_sentence'].str.len() < 150)
]

df = df.sample(min(len(df), 12000), random_state=42)           # reduce size
df = filter_len(df)                              # apply length filter

df = df.reset_index(drop=True)                   # reset index

print("Final dataset size:", df.shape)
df.head()

Final dataset size: (7703, 2)


,english_sentence,hindi_sentence
0,"They must also start to produce green energy,","उन्हें वैकल्पिक ऊर्जा का उत्पादन शुरू करना होगा,"
1,"In the U.S., in the Philippines, in Kenya, aro...","अमरीका में, फ़िलिपींस में, कीन्या में, सारे वि..."
2,It has a fine climate and vegetables are grown...,यहां की जलवायु बहुत अच्छी है और पर्याप्त मात्र...
3,India's Music and Dance style are also popular...,भारत में संगीत तथा नृत्य की अपनी शैलियां भी वि...
4,I was 19.,उस समय मैं 19 साल की थी.


In [38]:
import re

def clean_eng(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z?.!,¿]+", " ", text)
    return text.strip()

def clean_hin(text):
    return text.strip()

df["english_sentence"] = df["english_sentence"].apply(clean_eng)
df["hindi_sentence"] = df["hindi_sentence"].apply(lambda x: "<start> " + clean_hin(x) + " <end>")

In [39]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

eng_tok = Tokenizer(num_words=15000, oov_token="<unk>")
hin_tok = Tokenizer(num_words=15000, filters='', oov_token="<unk>")

eng_tok.fit_on_texts(df["english_sentence"])
hin_tok.fit_on_texts(df["hindi_sentence"])

eng_seq = eng_tok.texts_to_sequences(df["english_sentence"])
hin_seq = hin_tok.texts_to_sequences(df["hindi_sentence"])

max_len_eng = 15
max_len_hin = 15

eng_seq = pad_sequences(eng_seq, maxlen=max_len_eng, padding='post')
hin_seq = pad_sequences(hin_seq, maxlen=max_len_hin, padding='post')

In [40]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    eng_seq, hin_seq, test_size=0.1, random_state=42
)

In [41]:
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model

embedding_dim = 128
latent_dim = 256

# Encoder
encoder_inputs = Input(shape=(max_len_eng,))
enc_emb = Embedding(len(eng_tok.word_index)+1, embedding_dim)(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(None,))
dec_emb = Embedding(len(hin_tok.word_index)+1, embedding_dim)(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(len(hin_tok.word_index)+1, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [42]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(patience=15, restore_best_weights=True)

model.fit(
    [X_train, y_train[:, :-1]],
    y_train[:, 1:],
    batch_size=32,
    epochs=20,
    validation_split=0.1,
    callbacks=[early_stop]
)

Epoch 1/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 11s 37ms/step - accuracy: 0.2144 - loss: 6.8386 - val_accuracy: 0.3848 - val_loss: 4.8352
Epoch 2/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - accuracy: 0.3742 - loss: 4.6854 - val_accuracy: 0.3957 - val_loss: 4.7199
Epoch 3/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - accuracy: 0.3854 - loss: 4.4962 - val_accuracy: 0.4041 - val_loss: 4.6638
Epoch 4/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - accuracy: 0.3965 - loss: 4.3088 - val_accuracy: 0.4064 - val_loss: 4.6156
Epoch 5/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - accuracy: 0.4033 - loss: 4.1649 - val_accuracy: 0.4134 - val_loss: 4.5624
Epoch 6/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - accuracy: 0.4150 - loss: 3.9920 - val_accuracy: 0.4151 - val_loss: 4.5350
Epoch 7/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - accuracy: 0.4211 - loss: 3.8454 - val_accuracy: 0.4172 - val_loss: 4.5253
Epoch 8/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - accuracy: 0.4322 - loss: 3.6683 - val_acc

In [43]:
# Encoder inference
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder inference
from tensorflow.keras.layers import Input

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_emb2 = dec_emb
decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2, initial_state=decoder_states_inputs
)

decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + [state_h2, state_c2]
)

In [44]:
import numpy as np

index2word_hin = {v:k for k,v in hin_tok.word_index.items()}

def decode_sequence(input_seq):
    states = encoder_model.predict(input_seq, verbose=0)

    target_seq = np.zeros((1,1))
    target_seq[0,0] = hin_tok.word_index["<start>"]

    sentence = ""

    for _ in range(max_len_hin):
        output, h, c = decoder_model.predict([target_seq] + states, verbose=0)

        idx = np.argmax(output[0,-1,:])
        word = index2word_hin.get(idx, "")

        if word == "<end>":
            break

        sentence += " " + word

        target_seq = np.zeros((1,1))
        target_seq[0,0] = idx
        states = [h, c]

    return sentence.strip()

In [45]:
for i in range(5):
    print("ENG:", df["english_sentence"].iloc[i])
    print("PRED:", decode_sequence(X_test[i:i+1]))
    print("----")

ENG: they must also start to produce green energy,
PRED: <start> और ये एक तरह और एक एक एक साथ है।
----
ENG: in the u.s., in the philippines, in kenya, around the world,
PRED: <start> और एक एक एक एक एक एक एक साथ
----
ENG: it has a fine climate and vegetables are grown in abundance .
PRED: <start> और ये एक तरह के लिए एक साथ
----
ENG: india s music and dance style are also popular because of its vividness
PRED: <start> और ये एक एक एक एक एक एक साथ है।
----
ENG: i was .
PRED: <start> और एक एक एक एक एक एक साथ है।
----


TASK 2

In [46]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smooth = SmoothingFunction().method1
def evaluate(n=30):
    scores = []
    
    for i in range(n):
        pred = decode_sequence(X_test[i:i+1])
        
        actual_words = [
            index2word_hin.get(idx, "")
            for idx in y_test[i]
            if idx != 0
        ]

        ref = [actual_words]
        cand = pred.split()

        score = sentence_bleu(ref, cand, smoothing_function=smooth)
        scores.append(score)

    return sum(scores)/len(scores)

print("BLEU Score:", evaluate())

BLEU Score: 0.012300431544314157


TASK 3

In [47]:
!pip install datasets nltk

import numpy as np
import re
import tensorflow as tf
from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

In [48]:
dataset = load_dataset("opus_books", "en-es")

data = dataset["train"].shuffle(seed=42).select(range(15000))

README.md: 0.00B [00:00, ?B/s]

en-es/train-00000-of-00001.parquet:   0%|          | 0.00/16.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/93470 [00:00<?, ? examples/s]

In [49]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z?.!,¿]+", " ", text)
    return text.strip()

eng_texts = [clean_text(x["translation"]["en"]) for x in data]
spa_texts = [clean_text(x["translation"]["es"]) for x in data]

# Add special tokens AFTER cleaning
spa_texts = ["<start> " + x + " <end>" for x in spa_texts]

In [50]:
eng_tok = Tokenizer(num_words=10000, oov_token="<unk>")
spa_tok = Tokenizer(num_words=10000, filters='', oov_token="<unk>")

eng_tok.fit_on_texts(eng_texts)
spa_tok.fit_on_texts(spa_texts)

eng_seq = eng_tok.texts_to_sequences(eng_texts)
spa_seq = spa_tok.texts_to_sequences(spa_texts)

In [51]:
max_len_eng = 20
max_len_spa = 20

eng_seq = pad_sequences(eng_seq, maxlen=max_len_eng, padding='post')
spa_seq = pad_sequences(spa_seq, maxlen=max_len_spa, padding='post')

In [52]:
X_train, X_test, y_train, y_test = train_test_split(
    eng_seq, spa_seq, test_size=0.1, random_state=42
)

In [53]:
embedding_dim = 128
latent_dim = 256

# Encoder
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(10000, embedding_dim)(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)

encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(None,))
dec_emb = Embedding(10000, embedding_dim)(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(10000, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_9       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, None, 128) │  1,280,000 │ input_layer_8[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_7         │ (None, None, 128) │  1,280,000 │ input_layer_9[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_6 (LSTM)       │ [(None, 256),     │    394,240 │ embedding_6[0][0] │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_7 (LSTM)       │ [(None, None,     │    394,240 │ embedding_7[0][0… │
│                     │ 256), (None,      │            │ lstm_6[0][1],     │
│                     │ 256), (None,      │            │ lstm_6[0][2]      │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, None,      │  2,570,000 │ lstm_7[0][0]      │
│                     │ 10000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,918,480 (22.58 MB)

 Trainable params: 5,918,480 (22.58 MB)

 Non-trainable params: 0 (0.00 B)

In [54]:
history = model.fit(
    [X_train, y_train[:, :-1]],
    y_train[:, 1:],
    batch_size=32,
    epochs=15,
    validation_split=0.1
)

Epoch 1/15
380/380 ━━━━━━━━━━━━━━━━━━━━ 14s 30ms/step - accuracy: 0.2487 - loss: 5.7722 - val_accuracy: 0.3411 - val_loss: 4.4788
Epoch 2/15
380/380 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - accuracy: 0.3615 - loss: 4.2904 - val_accuracy: 0.3613 - val_loss: 4.2662
Epoch 3/15
380/380 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.3783 - loss: 4.0765 - val_accuracy: 0.3790 - val_loss: 4.1009
Epoch 4/15
380/380 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.4013 - loss: 3.8406 - val_accuracy: 0.3901 - val_loss: 4.0036
Epoch 5/15
380/380 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.4094 - loss: 3.7115 - val_accuracy: 0.3927 - val_loss: 3.9409
Epoch 6/15
380/380 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.4186 - loss: 3.5709 - val_accuracy: 0.3966 - val_loss: 3.9006
Epoch 7/15
380/380 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.4246 - loss: 3.4513 - val_accuracy: 0.3986 - val_loss: 3.8781
Epoch 8/15
380/380 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.4284 - loss: 3.3616 - 

In [55]:
# Encoder inference
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder inference
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_emb2 = dec_emb
decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2, initial_state=decoder_states_inputs
)

decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + [state_h2, state_c2]
)

In [56]:
index2word_spa = {v:k for k,v in spa_tok.word_index.items()}

def decode_sequence_spa(input_seq):
    states = encoder_model.predict(input_seq, verbose=0)

    target_seq = np.zeros((1,1))
    target_seq[0,0] = spa_tok.word_index["<start>"]

    sentence = ""

    for _ in range(max_len_spa):
        output, h, c = decoder_model.predict([target_seq] + states, verbose=0)

        idx = np.argmax(output[0,-1,:])
        word = index2word_spa.get(idx, "")

        if word == "<end>":
            break

        if word != "<unk>":
            sentence += " " + word

        target_seq[0,0] = idx
        states = [h, c]

    return sentence.strip()

In [57]:
for i in range(5):
    print("ENG:", eng_texts[i])
    print("PRED:", decode_sequence_spa(X_test[i:i+1]))
    print("----")

ENG: kitty, on the contrary, was more active than usual and even more animated.
PRED: el capit n nemo .
----
ENG: all necessary preparations shall be made for your return.
PRED: 
----
ENG: maintenant partons, allons, allons vers sion.
PRED: ¿c mo es
----
ENG: thursday, aug.
PRED: no me a mi y se or por el
----
ENG: richelieu
PRED: el capit n nemo se en el de la n de
----


In [58]:
smooth = SmoothingFunction().method1

def evaluate(n=50):
    scores = []
    
    for i in range(n):
        pred = decode_sequence_spa(X_test[i:i+1])
        
        actual_words = [
            index2word_spa.get(idx, "")
            for idx in y_test[i]
            if idx != 0 and index2word_spa.get(idx, "") not in ["<start>", "<end>"]
        ]
        
        ref = [actual_words]
        cand = pred.split()

        score = sentence_bleu(ref, cand, smoothing_function=smooth)
        scores.append(score)

    return sum(scores)/len(scores)

print("BLEU Score:", evaluate())

BLEU Score: 0.006700865642064409
